<a href="https://colab.research.google.com/github/ipeirotis/dealing_with_data/blob/master/01-Pandas/A1_Introduction_to_Pandas_Loading_and_Exploring%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A1: Introduction to Pandas — Loading and Exploring Data

## What is Pandas?

**Pandas** is Python's most popular library for data analysis. If you know SQL, you can think of Pandas as "SQL for Python"—but with superpowers for analysis, visualization, and machine learning integration.

The name "Pandas" comes from "Panel Data," a term from econometrics for multi-dimensional structured datasets.

## Why Use Pandas Instead of Just SQL?

Great question! You already know SQL, so why learn another tool?

| Task | SQL | Pandas |
|------|-----|--------|
| Store and query large datasets | ✅ Excellent | ⚠️ Limited by RAM |
| Filter, join, aggregate data | ✅ Yes | ✅ Yes |
| Statistical analysis | ⚠️ Limited | ✅ Excellent |
| Data visualization | ❌ No | ✅ Built-in |
| Machine learning integration | ❌ No | ✅ Seamless |
| Handle messy/unstructured data | ⚠️ Difficult | ✅ Flexible |
| Reproducible analysis scripts | ⚠️ Possible | ✅ Natural |

**The typical workflow**: Use SQL to query and filter data in the database, then bring a manageable subset into Pandas for analysis, visualization, and modeling.

## What You Will Learn

In this notebook, you will learn to:

1. **Load data** from a database into Pandas using SQL
2. **Understand the core data structures**: DataFrame and Series
3. **Explore data** using `.head()`, `.tail()`, `.shape`, `.dtypes`
4. **Generate descriptive statistics** for numeric and categorical variables
5. **Create basic visualizations**: histograms, bar charts, KDE plots

## The Dataset: NYC Restaurant Inspections

We'll work with real data from the NYC Department of Health: restaurant inspection results. Every restaurant in NYC is inspected and assigned a grade (A, B, C) based on violations found.

This dataset is:
- **Relatable**: Everyone eats at restaurants!
- **Rich**: Multiple tables, various data types, interesting patterns
- **Real**: Actual public data from NYC Open Data

You can explore the original dataset at: [NYC Open Data - Restaurant Inspections](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j)

---

## Part 1: Setup and Database Connection

First, we need to:
1. Authenticate with Google Cloud (to access BigQuery)
2. Import the libraries we'll use
3. Configure our plotting settings

### 🗳️ Quick Check-in

Before we dive in, let's see where everyone's at:

- **Show of hands**: How many of you have used Python before?
- **Show of hands**: How many have heard of Pandas?
- **Show of hands**: How many feel comfortable with SQL?

> 💡 **Good news**: Your SQL knowledge is your superpower here! Pandas is essentially "SQL for Python"—same concepts, new syntax.

In [ ]:
# ============================================================
# AUTHENTICATION - Run this cell and follow the login prompt
# ============================================================
from google.colab import auth
auth.authenticate_user()
print("✓ Authentication successful!")

In [ ]:
# ============================================================
# BIGQUERY SETUP
# ============================================================
!pip install -q google-cloud-bigquery
from google.cloud import bigquery

# ⚠️ IMPORTANT: Change this to your own Google Cloud project ID
# If you don't have one, ask your instructor for access
PROJECT_ID = "nyu-datasets"  # ← CHANGE THIS if needed

client = bigquery.Client(project=PROJECT_ID)
print(f"✓ Connected to BigQuery project: {PROJECT_ID}")

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================
import pandas as pd      # Data manipulation and analysis
import numpy as np       # Numerical computing
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns    # Statistical visualization

print(f"✓ Pandas version: {pd.__version__}")

In [ ]:
# ============================================================
# CONFIGURE PLOTTING (optional but makes plots look better)
# ============================================================
%config InlineBackend.figure_format = 'retina'  # High-resolution plots
plt.rcParams['figure.figsize'] = [10, 4]        # Default figure size
plt.rcParams['font.size'] = 11                  # Readable font size
sns.set_style("whitegrid")                      # Clean grid background

print("✓ Plotting configured!")

---

## Part 2: Loading Data from a Database

### The Bridge from SQL to Pandas

This is where your SQL knowledge pays off! We write a SQL query, execute it, and load the results directly into a Pandas DataFrame.

```python
# The pattern:
sql = "SELECT ... FROM ... WHERE ..."
df = client.query(sql).to_dataframe()
```

The `.to_dataframe()` method converts the BigQuery result into a Pandas DataFrame.

### Our Database Schema

The restaurant inspection data is stored in three tables:

| Table | Description | Key Columns |
|-------|-------------|-------------|
| `restaurants` | Restaurant info | CAMIS (ID), DBA (name), BORO, CUISINE_DESCRIPTION |
| `inspections` | Inspection results | INSPECTION_DATE, SCORE, GRADE |
| `violations` | Specific violations | VIOLATION_CODE, VIOLATION_DESCRIPTION |

**CAMIS** is the unique restaurant identifier (like a primary key).

In [ ]:
# Load restaurant information
sql = """
SELECT
    CAMIS,                    -- Unique restaurant ID
    DBA,                      -- "Doing Business As" (restaurant name)
    BUILDING,                 -- Street number
    STREET,                   -- Street name
    ZIPCODE,
    BORO,                     -- Borough (Manhattan, Brooklyn, etc.)
    CUISINE_DESCRIPTION,      -- Type of cuisine
    LATITUDE,
    LONGITUDE
FROM `nyu-datasets.doh_restaurants.restaurants`
"""

restaurants = client.query(sql).to_dataframe()
print(f"✓ Loaded {len(restaurants):,} restaurants")

In [ ]:
# Load inspection results (joining restaurants and inspections tables)
sql = """
SELECT
    R.CAMIS,
    R.DBA,
    R.ZIPCODE,
    R.BORO,
    R.CUISINE_DESCRIPTION,
    I.INSPECTION_DATE,
    I.INSPECTION_TYPE,
    I.SCORE,
    I.GRADE
FROM `nyu-datasets.doh_restaurants.restaurants` R
JOIN `nyu-datasets.doh_restaurants.inspections` I ON I.CAMIS = R.CAMIS
"""

inspections = client.query(sql).to_dataframe()
print(f"✓ Loaded {len(inspections):,} inspection records")

### 🎯 Activity 1: SQL-to-Pandas Bridge

Now it's your turn! Use your SQL skills to create filtered dataframes.

**Task**: Create two new dataframes:
1. `manh_restaurants` — containing ONLY restaurants in Manhattan
2. `manh_inspections` — containing ONLY inspections for Manhattan restaurants

**Hint**: Use `WHERE BORO = 'Manhattan'` in your SQL query

**Expected results**:
- ~8,200 Manhattan restaurants
- ~22,700 Manhattan inspections

In [ ]:
# Activity 1: Create Manhattan-only dataframes
# YOUR CODE HERE

# sql = """
# SELECT ...
# FROM ...
# WHERE ...
# """
# manh_restaurants = client.query(sql).to_dataframe()

# TODO: Also create manh_inspections


---

## Part 3: Understanding DataFrames and Series

### The Two Core Data Structures

Pandas has two main data structures:

| Structure | What It Is | SQL Equivalent |
|-----------|------------|----------------|
| **DataFrame** | A 2D table with rows and columns | A table or query result |
| **Series** | A single column of data | One column from a table |

Think of it this way:
- A **DataFrame** is like an Excel spreadsheet or a SQL table
- A **Series** is like a single column from that spreadsheet

### Viewing a DataFrame

In [ ]:
# Display the restaurants DataFrame
# By default, Jupyter shows the first and last few rows
display(restaurants)

**Notice three things:**

1. **The Index** (leftmost column, in bold): Row labels (0, 1, 2, ...). Like an automatic row ID.
2. **Column Headers**: The names of each column (CAMIS, DBA, BUILDING, ...)
3. **The Data**: The actual values in each cell

### Viewing Parts of a DataFrame

In [ ]:
# .head(n) shows the first n rows (default: 5)
restaurants.head(3)

In [ ]:
# .tail(n) shows the last n rows
restaurants.tail(3)

In [ ]:
# .sample(n) shows n random rows — great for getting a feel for the data
restaurants.sample(5)

### 🎯 Activity 2: DataFrame Exploration Challenge

**Speed challenge!** Answer these questions as quickly as you can:

1. How many rows are in the `inspections` dataframe? (Use `.shape`)
2. What are all the column names in `inspections`? (Use `.columns`)
3. Display 3 random inspections (Use `.sample()`)
4. What restaurant is in the very last row of `restaurants`? (Use `.tail(1)`)

*Type your answers in the code cell below:*

In [ ]:
# Activity 2: Answer the exploration questions

# 1. Number of rows in inspections:


# 2. Column names in inspections:


# 3. Three random inspections:


# 4. Last row of restaurants:



### Extracting a Single Column (Series)

In [ ]:
# Use bracket notation to extract a single column
# This returns a Series (not a DataFrame)
restaurants["DBA"]

In [ ]:
# Check the type
print(f"restaurants is a:        {type(restaurants)}")
print(f"restaurants['DBA'] is a: {type(restaurants['DBA'])}")

### 🎯 Exercise 1: Explore the Inspections DataFrame

1. Display the `inspections` DataFrame
2. Show the first 10 rows
3. Show 5 random rows
4. Extract the `GRADE` column as a Series

In [ ]:
# Your code here


---

## Part 4: DataFrame Metadata

Before analyzing data, you should always understand its structure:
- How many rows and columns?
- What are the column names?
- What data types are in each column?

### Size and Shape

In [ ]:
# .shape returns (rows, columns) as a tuple
print(f"Restaurants: {restaurants.shape[0]:,} rows × {restaurants.shape[1]} columns")
print(f"Inspections: {inspections.shape[0]:,} rows × {inspections.shape[1]} columns")

In [ ]:
# len() gives just the number of rows
print(f"Number of restaurants: {len(restaurants):,}")

### Column Names

In [ ]:
# .columns shows all column names
restaurants.columns

In [ ]:
# Convert to a list for easier reading
list(restaurants.columns)

### Data Types

Understanding data types is crucial because different operations work on different types:

| Pandas dtype | Description | Example Values |
|--------------|-------------|----------------|
| `int64` | Integer numbers | 1, 42, -7 |
| `float64` | Decimal numbers | 3.14, -2.5 |
| `object` | Text/strings (or mixed) | "Hello", "NYC" |
| `datetime64` | Dates and times | 2024-01-15 |
| `bool` | True/False | True, False |

In [ ]:
# .dtypes shows the data type of each column
restaurants.dtypes

In [ ]:
# Check inspections data types — notice INSPECTION_DATE is datetime!
inspections.dtypes

### 🎯 Activity 3: Data Type Detective

Understanding data types is crucial for analysis. Let's investigate!

**Tasks**:
1. Check the data types of the `inspections` dataframe using `.dtypes`
2. Answer these questions:
   - What type is `INSPECTION_DATE`? Why is this useful?
   - What type is `ZIPCODE`? Should it be numeric?
   - Which columns could you calculate a mean for?

**Discussion**: Why is ZIPCODE stored as text (`object`) instead of a number? *Think: Would it make sense to calculate the average ZIP code?*

In [ ]:
# Activity 3: Investigate data types

# Check inspections data types:


# Your observations:
# - INSPECTION_DATE is type: ___________
# - ZIPCODE is type: ___________
# - Columns I can calculate mean for: ___________


### Quick Summary with `.info()`

In [ ]:
# .info() gives a comprehensive overview:
# - Number of rows and columns
# - Column names and types
# - Non-null counts (helps identify missing data)
# - Memory usage
restaurants.info()

---

## Part 5: Descriptive Statistics

Now let's explore what's actually *in* the data. We use different techniques for different data types:

| Data Type | Key Methods |
|-----------|-------------|
| Numeric (int, float) | `.describe()`, `.mean()`, `.median()`, `.hist()` |
| Categorical (object) | `.value_counts()`, `.nunique()`, bar charts |
| DateTime | `.min()`, `.max()`, time-based histograms |

### Numeric Variables: The SCORE Column

In [ ]:
# .describe() gives summary statistics for numeric columns
inspections["SCORE"].describe()

**Interpretation:**
- **count**: Number of non-null values
- **mean**: Average score (~13)
- **std**: Standard deviation (spread of data)
- **min/max**: Range of values
- **25%, 50%, 75%**: Quartiles (50% is the median)

**Note about SCORE**: Lower is better! A score of 0-13 = Grade A, 14-27 = Grade B, 28+ = Grade C.

In [ ]:
# Individual statistics
print(f"Mean score:   {inspections['SCORE'].mean():.2f}")
print(f"Median score: {inspections['SCORE'].median():.2f}")
print(f"Std dev:      {inspections['SCORE'].std():.2f}")
print(f"Min score:    {inspections['SCORE'].min()}")
print(f"Max score:    {inspections['SCORE'].max()}")

### Visualizing Numeric Data: Histograms

In [ ]:
# Basic histogram
inspections["SCORE"].hist()

In [ ]:
# Better histogram with more bins and limited range
inspections["SCORE"].hist(
    bins=50,           # More bars = more detail
    range=(0, 50),     # Focus on typical scores (0-50)
    edgecolor='white', # White borders between bars
)
plt.xlabel("Inspection Score")
plt.ylabel("Number of Inspections")
plt.title("Distribution of NYC Restaurant Inspection Scores")
plt.axvline(x=13, color='green', linestyle='--', label='A/B threshold (13)')
plt.axvline(x=28, color='orange', linestyle='--', label='B/C threshold (28)')
plt.legend()

### 🎯 Activity 4: Histogram Customization Challenge

The basic histogram above doesn't tell the full story. Let's make it better!

**Tasks**:
1. Create a histogram of SCORE with:
   - `bins=50` for more detail
   - `range=(0, 50)` to focus on typical scores
   - A descriptive title
   
2. **Research question**: What score range gets an 'A' grade? (Hint: Google "NYC restaurant inspection scoring")

**Bonus**: Try adding `cumulative=True` — what does this show you?

In [ ]:
# Activity 4: Create an improved histogram

# Your histogram code here:


# What score range gets an A grade? ___________


**Observation**: Most restaurants score below 13 (Grade A range). The distribution is right-skewed with some high outliers.

### Kernel Density Estimation (KDE)

A KDE plot is a smoothed version of a histogram—it estimates the probability density function.

In [ ]:
# KDE plot
inspections["SCORE"].plot(kind='kde', xlim=(0, 50))
plt.xlabel("Inspection Score")
plt.title("Density of Inspection Scores")

In [ ]:
# Combine histogram and KDE
ax = inspections["SCORE"].hist(bins=50, range=(0, 50), density=True, alpha=0.7, edgecolor='white')
inspections["SCORE"].plot(kind='kde', xlim=(0, 50), ax=ax, color='black', linewidth=2)
plt.xlabel("Inspection Score")
plt.title("Distribution of Inspection Scores (Histogram + KDE)")

### Categorical Variables: value_counts()

For text/categorical columns, we want to know: What are the unique values and how often does each appear?

In [ ]:
# Count restaurants by borough
restaurants["BORO"].value_counts()

In [ ]:
# Count by cuisine type (top 15)
restaurants["CUISINE_DESCRIPTION"].value_counts().head(15)

In [ ]:
# How many unique cuisines are there?
print(f"Number of unique cuisine types: {restaurants['CUISINE_DESCRIPTION'].nunique()}")

### Visualizing Categorical Data: Bar Charts

In [ ]:
# Horizontal bar chart of top 10 cuisines
# (horizontal because labels are easier to read)
(
    restaurants["CUISINE_DESCRIPTION"]
    .value_counts()
    .head(10)
    .sort_values()  # Sort ascending so largest is at top
    .plot(kind='barh', color='steelblue')
)
plt.xlabel("Number of Restaurants")
plt.title("Top 10 Cuisine Types in NYC")

In [ ]:
# Grade distribution
inspections["GRADE"].value_counts().sort_index().plot(kind='bar', color=['green', 'gold', 'red'])
plt.xlabel("Grade")
plt.ylabel("Number of Inspections")
plt.title("Distribution of Inspection Grades")
plt.xticks(rotation=0)

### DateTime Variables

The `INSPECTION_DATE` column is a datetime type, which enables powerful time-based analysis.

In [ ]:
# Date range of the data
print(f"Earliest inspection: {inspections['INSPECTION_DATE'].min()}")
print(f"Latest inspection:   {inspections['INSPECTION_DATE'].max()}")

In [ ]:
# Histogram of inspection dates
inspections["INSPECTION_DATE"].hist(bins=100, edgecolor='white')
plt.xlabel("Date")
plt.ylabel("Number of Inspections")
plt.title("Inspections Over Time")

### 🎯 Activity 5: Restaurant Chain Analysis

The `DBA` column contains restaurant names. Chains like Dunkin' and Starbucks appear multiple times.

**Tasks**:
1. Find the top 15 most common restaurant names (chains)
2. Create a horizontal bar chart of these chains
3. **Comparison**: How does the list change if you only look at Manhattan? (Use your `manh_restaurants` dataframe)

**Code pattern**:
```python
df["COLUMN"].value_counts().head(15).plot(kind="barh")
```

In [ ]:
# Activity 5: Analyze restaurant chains

# 1. Top 15 restaurant names in all of NYC:


# 2. Create a bar chart:


# 3. Compare to Manhattan only (if you created manh_restaurants):



### 🎯 Activity 6: Violation Code Investigation (Challenge)

Let's load and analyze violation data! This combines SQL querying with Pandas analysis.

**Tasks**:
1. Run the query below to load the violations dataframe
2. Find the 10 most common violation codes
3. Create a bar chart showing their frequency
4. **Bonus**: Query `doh_restaurants.violation_codes` to get the descriptions!

**Discussion**: What types of violations are most common? Are these serious or minor?

In [ ]:
# Activity 6: Load violations data
sql = '''
WITH latest_inspection AS (
    SELECT CAMIS, MAX(INSPECTION_DATE) AS INSPECTION_DATE
    FROM `nyu-datasets.doh_restaurants.inspections`
    GROUP BY CAMIS
)
SELECT R.CAMIS, R.DBA, R.ZIPCODE, R.BORO,
       I.INSPECTION_DATE, I.SCORE, I.GRADE,
       V.VIOLATION_CODE
FROM `nyu-datasets.doh_restaurants.restaurants` R
    JOIN latest_inspection L ON R.CAMIS = L.CAMIS
    JOIN `nyu-datasets.doh_restaurants.inspections` I
        ON I.CAMIS = L.CAMIS AND L.INSPECTION_DATE = I.INSPECTION_DATE
    JOIN `nyu-datasets.doh_restaurants.violations` V
        ON I.INSPECTION_ID = V.INSPECTION_ID
'''

violations = client.query(sql).to_dataframe()
print(f"✓ Loaded {len(violations):,} violation records")

In [ ]:
# Activity 6: Analyze violations

# 1. Find top 10 most common violation codes:


# 2. Create a bar chart:


# 3. BONUS: Query violation_codes table to get descriptions:
# sql = "SELECT * FROM `nyu-datasets.doh_restaurants.violation_codes`"
# codes = client.query(sql).to_dataframe()



### 🎯 Activity 7: Geographic Comparison

Let's compare restaurants across NYC's five boroughs.

**Tasks**:
1. Create a bar chart showing the number of restaurants per borough
2. Create a bar chart showing the number of restaurants per ZIP code (top 15)
3. **Analysis**: Which borough has the most restaurants? Which ZIP code?

In [ ]:
# Activity 7: Geographic comparison

# 1. Restaurants per borough:


# 2. Restaurants per ZIP code (top 15):


# 3. Your observations:
# Most restaurants in borough: ___________
# Busiest ZIP code: ___________


---

## Part 6: Geographic Exploration (Preview)

Since we have latitude and longitude data, let's create a quick geographic visualization. This is a preview of techniques we'll explore more deeply in the spatial data module.

In [ ]:
# Simple scatter plot of restaurant locations
restaurants.plot(
    kind='scatter',
    x='LONGITUDE',
    y='LATITUDE',
    alpha=0.1,      # Transparency (many overlapping points)
    s=1,            # Small dot size
    figsize=(8, 8),
    title="Restaurant Locations in NYC"
)

**Observation**: You can see the shape of NYC! The dense cluster at the center is Manhattan, with downtown having higher density. The large area to the right is Brooklyn and Queens.

---

## Summary: What You Learned

### Core Concepts

| Concept | Description |
|---------|-------------|
| **DataFrame** | A 2D table (like a SQL result set) |
| **Series** | A single column from a DataFrame |
| **Index** | Row labels (usually 0, 1, 2, ...) |
| **dtypes** | Data types of each column |

### Key Methods

| Method | Purpose | Example |
|--------|---------|--------|
| `.head(n)` | View first n rows | `df.head(10)` |
| `.tail(n)` | View last n rows | `df.tail(5)` |
| `.sample(n)` | View n random rows | `df.sample(10)` |
| `.shape` | Get (rows, columns) | `df.shape` |
| `.columns` | Get column names | `df.columns` |
| `.dtypes` | Get data types | `df.dtypes` |
| `.info()` | Comprehensive overview | `df.info()` |
| `.describe()` | Numeric statistics | `df['col'].describe()` |
| `.value_counts()` | Count unique values | `df['col'].value_counts()` |
| `.nunique()` | Count distinct values | `df['col'].nunique()` |
| `.hist()` | Create histogram | `df['col'].hist(bins=50)` |
| `.plot()` | Create various plots | `df['col'].plot(kind='bar')` |

### What's Next?

In **Notebook A2**, you'll learn to manipulate data:
- Selecting rows (like SQL `WHERE`)
- Selecting columns (like SQL `SELECT`)
- Sorting, filtering, and joining tables
- Grouping and aggregating (like SQL `GROUP BY`)

---

# 📝 Activity Solutions

**Note**: Try to complete the activities on your own first! Solutions are here for reference and self-checking.

---

In [ ]:
# =============================================================================
# SOLUTION: Activity 1 - SQL-to-Pandas Bridge
# =============================================================================

# Manhattan restaurants
sql = """
SELECT CAMIS, DBA, BUILDING, STREET, ZIPCODE, BORO,
       CUISINE_DESCRIPTION, LATITUDE, LONGITUDE
FROM `nyu-datasets.doh_restaurants.restaurants`
WHERE BORO = 'Manhattan'
"""
manh_restaurants = client.query(sql).to_dataframe()
print(f"Manhattan restaurants: {len(manh_restaurants):,}")

# Manhattan inspections
sql = """
SELECT R.CAMIS, R.DBA, R.ZIPCODE, R.BORO, R.CUISINE_DESCRIPTION,
       I.INSPECTION_DATE, I.INSPECTION_TYPE, I.SCORE, I.GRADE
FROM `nyu-datasets.doh_restaurants.restaurants` R
JOIN `nyu-datasets.doh_restaurants.inspections` I ON I.CAMIS = R.CAMIS
WHERE R.BORO = 'Manhattan'
"""
manh_inspections = client.query(sql).to_dataframe()
print(f"Manhattan inspections: {len(manh_inspections):,}")

In [ ]:
# =============================================================================
# SOLUTION: Activity 2 - DataFrame Exploration Challenge
# =============================================================================

# 1. Number of rows in inspections:
print(f"Inspections has {inspections.shape[0]:,} rows")

# 2. Column names:
print(f"\nColumns: {list(inspections.columns)}")

# 3. Three random inspections:
print("\n3 random inspections:")
display(inspections.sample(3))

# 4. Last row:
print("\nLast restaurant:")
display(restaurants.tail(1))

In [ ]:
# =============================================================================
# SOLUTION: Activity 3 - Data Type Detective
# =============================================================================

print("Inspections data types:")
print(inspections.dtypes)

print("\n" + "="*50)
print("ANSWERS:")
print("="*50)
print("- INSPECTION_DATE is: datetime64 (allows date math and filtering)")
print("- ZIPCODE is: object (string) - correct! Can't average ZIP codes")
print("- Can calculate mean for: SCORE (and technically LATITUDE/LONGITUDE)")

In [ ]:
# =============================================================================
# SOLUTION: Activity 4 - Histogram Customization
# =============================================================================

# Improved histogram
inspections["SCORE"].plot.hist(
    bins=50,
    range=(0, 50),
    figsize=(10, 4),
    edgecolor='white',
    alpha=0.7,
    title="Distribution of NYC Restaurant Inspection Scores"
)
plt.xlabel("Inspection Score (lower is better)")
plt.ylabel("Number of Inspections")

# Add grade zone annotations
plt.axvline(x=13, color='green', linestyle='--', label='A/B boundary (13)')
plt.axvline(x=27, color='orange', linestyle='--', label='B/C boundary (27)')
plt.legend()

print("NYC Restaurant Grading:")
print("- Grade A: 0-13 points")
print("- Grade B: 14-27 points")
print("- Grade C: 28+ points")

In [ ]:
# =============================================================================
# SOLUTION: Activity 5 - Restaurant Chain Analysis
# =============================================================================

# 1. Top 15 restaurant names in all of NYC
print("Top 15 Restaurant Chains in NYC:")
top_chains = restaurants["DBA"].value_counts().head(15)
print(top_chains)

# 2. Bar chart
plt.figure(figsize=(10, 6))
top_chains.sort_values().plot(kind="barh")
plt.xlabel("Number of Locations")
plt.title("Top 15 Restaurant Chains in NYC")
plt.tight_layout()

# 3. Compare to Manhattan (if manh_restaurants exists)
try:
    print("\n" + "="*50)
    print("Top 15 Restaurant Chains in MANHATTAN:")
    print(manh_restaurants["DBA"].value_counts().head(15))
except NameError:
    print("\n(Run Activity 1 first to create manh_restaurants)")

In [ ]:
# =============================================================================
# SOLUTION: Activity 6 - Violation Code Investigation
# =============================================================================

# 1. Top 10 most common violation codes
print("Top 10 Most Common Violation Codes:")
top_violations = violations["VIOLATION_CODE"].value_counts().head(10)
print(top_violations)

# 2. Bar chart
plt.figure(figsize=(10, 5))
top_violations.sort_values().plot(kind="barh")
plt.xlabel("Number of Violations")
plt.title("Most Common Violation Codes in NYC Restaurants")
plt.tight_layout()

# 3. BONUS: Get violation descriptions
print("\n" + "="*50)
print("Loading violation code descriptions...")
sql = "SELECT * FROM `nyu-datasets.doh_restaurants.violation_codes`"
codes = client.query(sql).to_dataframe()
print(f"\nTop violation codes with descriptions:")
for code in top_violations.head(10).index:
    desc = codes[codes['VIOLATION_CODE'] == code]['DESCRIPTION'].values
    if len(desc) > 0:
        print(f"\n{code}: {desc[0][:100]}...")

In [ ]:
codes

In [ ]:
# =============================================================================
# SOLUTION: Activity 7 - Geographic Comparison
# =============================================================================

# 1. Restaurants per borough
plt.figure(figsize=(8, 4))
restaurants["BORO"].value_counts().sort_values().plot(kind="barh")
plt.xlabel("Number of Restaurants")
plt.title("Restaurants by Borough")
plt.tight_layout()
plt.show()

# 2. Restaurants per ZIP code (top 15)
plt.figure(figsize=(10, 5))
restaurants["ZIPCODE"].value_counts().head(15).sort_values().plot(kind="barh")
plt.xlabel("Number of Restaurants")
plt.title("Top 15 ZIP Codes by Restaurant Count")
plt.tight_layout()
plt.show()

# Analysis
print("\n" + "="*50)
print("ANALYSIS:")
print("="*50)
print(f"Borough with most restaurants: {restaurants['BORO'].value_counts().idxmax()}")
print(f"ZIP code with most restaurants: {restaurants['ZIPCODE'].value_counts().idxmax()}")